# Final Project Deliverable: Detection and Longitudinal Evolution of AI-Generated Scientific Text 

**Course:** Deep Learning for Natural Language Processing (NLP)  
**Academic Semester:** 2026/2  
**Author:** Gustavo Freitas Cunha  

---

> **AI Disclosure and Academic Integrity Statement**  
> In accordance with institutional policies on the ethical use of Artificial Intelligence (https://www.ufmg.br/ia/), this deliverable documents the utilization of AI assistance:  
>  
> - **Agents Utilized:** Google Antigravity (agentic assistant powered by Gemini).  
> - **Scope of AI Assistance:** Used strictly for code boilerplate generation, documentation scaffolding, and syntax formatting.  
> - **Student Authorship and Oversight:** All conceptual formulations, architectural choices, experimental workflows, and analytical interpretations were designed and authored directly by me.  
> - **Validation:** Every pipeline component and result was reviewed, compiled, and validated by me in the runtime environment.

# 1. Setup & Environment

In this section, I install required NLP dependencies, centralize scientific module imports, fix deterministic seeds for experimental reproducibility, and inspect the runtime hardware accelerator (Nvidia Tesla T4 GPU on Google Colab).

In [1]:
#CELL 1.1 - INSTALL REQUIRED LIBRARIES AND NLP DEPENDENCIES
%pip install -q transformers datasets evaluate scikit-learn optuna spacy torch matplotlib seaborn pandas numpy
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
#CELL 1.2 - IMPORT LIBRARIES AND CONFIGURE REPRODUCIBILITY ENVIRONMENT
import os
import sys
import gc
import json
import gzip
import urllib.request
import time
import random
import inspect
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

import spacy
import optuna

#configure deterministic random seeds across libraries
GLOBAL_SEED = 42

def configure_environment(seed: int = GLOBAL_SEED) -> None:
    #fix random seeds for scientific reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

configure_environment(GLOBAL_SEED)

#configure publication quality visual aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
})

print(f"Reproducibility environment configured successfully with random seed {GLOBAL_SEED}.")


Reproducibility environment configured successfully with random seed 42.


In [3]:
#CELL 1.3 - INSPECT RUNTIME ACCELERATOR AND ENVIRONMENT CONFIGURATION
def inspect_hardware_environment() -> Dict[str, Any]:
    #query system platform cuda driver and available gpu resources
    has_cuda = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if has_cuda else "CPU (No CUDA Accelerator)"
    device_count = torch.cuda.device_count() if has_cuda else 0
    total_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if has_cuda else 0.0
    )

    environment_info = {
        "python_version": sys.version.split()[0],
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "cuda_available": has_cuda,
        "device_name": device_name,
        "device_count": device_count,
        "total_memory_gb": round(total_memory_gb, 2),
    }

    print("=" * 70)
    print("RUNTIME HARDWARE ACCELERATOR INSPECTION")
    print(f"Python Version:       {environment_info['python_version']}")
    print(f"PyTorch Version:      {environment_info['torch_version']}")
    print(f"Transformers Version: {environment_info['transformers_version']}")
    print(f"CUDA Available:       {environment_info['cuda_available']}")
    print(f"Device Name:          {environment_info['device_name']}")
    print(f"Device Count:         {environment_info['device_count']}")
    if has_cuda:
        print(f"VRAM Capacity:        {environment_info['total_memory_gb']} GB")
    print("=" * 70)
    return environment_info

_ = inspect_hardware_environment()

RUNTIME HARDWARE ACCELERATOR INSPECTION
Python Version:       3.13.18
PyTorch Version:      2.14.1+cpu
Transformers Version: 5.22.1
CUDA Available:       False
Device Name:          CPU (No CUDA Accelerator)
Device Count:         0


# 2. Data Acquisition & Corpus Overview

This section establishes the empirical foundation for detecting machine-generated scientific text. The corpus comprises peer-reviewed papers published between 2015 and 2025 across four primary venues in artificial intelligence and computational linguistics: ACL, NeurIPS, ICML, and ICLR.

### Data Engineering and Preprocessing Pipeline Summary

1. **Stratified Web Scraping and In-Memory Extraction:**  
   The primary collection pipeline was executed via [collector.py](https://github.com/gustavofcunha/deep-learning-for-nlp-project/blob/main/src/coleta/collector.py). Rather than storing massive PDF binaries, the script streamed PDFs directly into memory buffers, parsing only the relevant structural components (Abstract, Introduction, and Conclusion) through PyMuPDF, while verifying strict quality gates (English language validation, minimum word thresholds, and SHA-256 deduplication).

2. **Specialized Text Preprocessing and Shortcut Neutralization:**  
   To prevent neural models from learning spurious formatting artifacts (shortcut learning) rather than authentic linguistic style, all texts underwent a deterministic 13-stage cleaning pipeline via [preprocessor.py](https://github.com/gustavofcunha/deep-learning-for-nlp-project/blob/main/src/preprocessing/preprocessor.py). Key stages include:
   - Canonical Unicode normalization (NFKC) and dehyphenation across line endings.
   - Section boundary truncation (pruning bibliographies, appendices, and post-text metadata).
   - Expurge of algorithm pseudocode listings, figure and table captions, and author email/affiliation footnotes.
   - Canonical shortcut masking: replacing citations with `[CITE]`, mathematical equations with `[MATH]`, and web links with `[URL]`.

3. **Two-Stage Synthetic Mirror Generation:**  
   Using 1,500 human seed papers from the pre-2020 partition (`provenance_type: 'seed_discarded'`), synthetic counterparts were generated using `Qwen/Qwen2.5-7B-Instruct` deployed on a dual Nvidia Tesla T4 cluster. A two-stage prompt workflow was employed:
   - **Stage A (Factual Extraction):** Distilling problem statements, methodologies, baselines, and empirical results.
   - **Stage B (Academic Mirror Drafting):** Synthesizing complete Abstract, Introduction, and Conclusion sections mirroring authentic scientific prose.
   - **Parity Preprocessing:** Synthetic outputs were sanitized locally using the exact same preprocessing pipeline, resulting in 1,494 fully masked synthetic papers (`label: 1`).

### Cloud and Local Execution Environment Support

While raw collected files (128 MB) and uncompressed JSONL dumps (~96 MB) remain outside Git history, compact gzip-compressed versions of the preprocessed corpora (`preprocessed_corpus.jsonl.gz` at 26.8 MB and `synthetic_corpus.jsonl.gz` at 1.4 MB) are hosted in `data/processed/` on GitHub. When executing Cell 2.1:
- **Local Workspace:** Automatically resolves local files across the repository directory tree without network requests.
- **Google Colab / Cloud Runtimes:** Automatically fetches the two compressed `.jsonl.gz` files directly from GitHub into `/content/data/processed/` with zero manual intervention (no Drive mounts or upload prompts required).

All collection code, preprocessing modules, and documentation are openly available at:  
[https://github.com/gustavofcunha/deep-learning-for-nlp-project](https://github.com/gustavofcunha/deep-learning-for-nlp-project)


In [4]:
#CELL 2.1 - CONFIGURE DATA PATHS AND AUTOMATED GITHUB RECOVERY
GITHUB_RAW_BASE_URL = (
    "https://raw.githubusercontent.com/gustavofcunha/deep-learning-for-nlp-project/main/data/processed"
)

def _download_corpus_file(url: str, dest_path: Path) -> bool:
    #stream remote gzipped corpus file atomically with basic integrity validation
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".tmp")
    try:
        print(f"Downloading {dest_path.name} from GitHub...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=120) as response, open(tmp_path, "wb") as out_file:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out_file.write(chunk)
        #verify gzip stream header before committing downloaded file
        if str(dest_path).endswith(".gz"):
            with gzip.open(tmp_path, "rb") as gz_check:
                gz_check.read(32)
        tmp_path.replace(dest_path)
        size_mb = dest_path.stat().st_size / (1024 * 1024)
        print(f"  Saved {dest_path} ({size_mb:.2f} MB)")
        return True
    except Exception as download_err:
        if tmp_path.exists():
            tmp_path.unlink()
        print(f"  Download failed for {url}: {download_err}")
        return False

def _locate_existing_datasets(search_dirs: List[Path]) -> Tuple[Optional[Path], Optional[Path]]:
    #search ordered candidate directories for uncompressed or compressed dataset files
    human_path: Optional[Path] = None
    synthetic_path: Optional[Path] = None
    for base_dir in search_dirs:
        if not base_dir.exists():
            continue
        for human_name in ("preprocessed_corpus.jsonl", "preprocessed_corpus.jsonl.gz"):
            cand = base_dir / human_name
            if cand.exists() and human_path is None:
                human_path = cand
                break
        for syn_name in ("synthetic_corpus.jsonl", "synthetic_corpus.jsonl.gz"):
            cand = base_dir / syn_name
            if cand.exists() and synthetic_path is None:
                synthetic_path = cand
                break
        if human_path and synthetic_path:
            break
    return human_path, synthetic_path

def resolve_data_paths(custom_dir: Optional[str] = None) -> Tuple[Path, Path]:
    #locate preprocessed datasets locally or fetch compressed mirrors from github
    search_dirs: List[Path] = []
    if custom_dir:
        search_dirs.append(Path(custom_dir))
    if os.environ.get("CORPUS_DATA_DIR"):
        search_dirs.append(Path(os.environ["CORPUS_DATA_DIR"]))

    #standard candidates across local workspace and colab layouts
    search_dirs.extend([
        Path.cwd() / "data" / "processed",
        Path.cwd().parent / "data" / "processed",
        Path.cwd().parent.parent / "data" / "processed",
        Path.home() / "Projeto-Final-APPNL" / "data" / "processed",
        Path.home() / "deep-learning-for-nlp-project" / "data" / "processed",
        Path("/content/data/processed"),
        Path("/content"),
        Path("/content/deep-learning-for-nlp-project/data/processed"),
        Path("/content/Projeto-Final-APPNL/data/processed"),
        Path("/content/drive/MyDrive/Projeto-Final-APPNL/data/processed"),
        Path("/content/drive/MyDrive/deep-learning-for-nlp-project/data/processed"),
        Path("/content/drive/MyDrive/data/processed"),
    ])

    #traverse upward directory tree up to 4 levels from current working directory
    current_path = Path.cwd().resolve()
    for _ in range(4):
        search_dirs.append(current_path / "data" / "processed")
        search_dirs.append(current_path)
        if current_path.parent == current_path:
            break
        current_path = current_path.parent

    #scan dynamically mounted google drive paths if available
    drive_base = Path("/content/drive/MyDrive")
    if drive_base.exists():
        try:
            for matched_file in drive_base.glob("**/preprocessed_corpus.jsonl*"):
                search_dirs.append(matched_file.parent)
        except Exception:
            pass

    human_corpus_path, synthetic_corpus_path = _locate_existing_datasets(search_dirs)

    #download compressed datasets from github when missing in remote runtimes
    if human_corpus_path is None or synthetic_corpus_path is None:
        target_dir = Path("/content/data/processed") if Path("/content").exists() else (Path.cwd() / "data" / "processed")
        target_dir.mkdir(parents=True, exist_ok=True)
        print("Datasets not found locally. Fetching compressed files from GitHub repository...")

        if human_corpus_path is None:
            dest_human = target_dir / "preprocessed_corpus.jsonl.gz"
            if _download_corpus_file(f"{GITHUB_RAW_BASE_URL}/preprocessed_corpus.jsonl.gz", dest_human):
                human_corpus_path = dest_human

        if synthetic_corpus_path is None:
            dest_synthetic = target_dir / "synthetic_corpus.jsonl.gz"
            if _download_corpus_file(f"{GITHUB_RAW_BASE_URL}/synthetic_corpus.jsonl.gz", dest_synthetic):
                synthetic_corpus_path = dest_synthetic

    if human_corpus_path is None or synthetic_corpus_path is None:
        raise FileNotFoundError(
            "Could not locate or download preprocessed_corpus.jsonl.gz and synthetic_corpus.jsonl.gz.\n"
            "Verify that the compressed files are pushed to the main branch on GitHub:\n"
            f"  {GITHUB_RAW_BASE_URL}/preprocessed_corpus.jsonl.gz\n"
            f"  {GITHUB_RAW_BASE_URL}/synthetic_corpus.jsonl.gz"
        )

    print("=" * 70)
    print("RESOLVED CORPUS FILE PATHS")
    print(f"  Human Corpus File:    {human_corpus_path}")
    print(f"  Synthetic Corpus File:{synthetic_corpus_path}")
    print("=" * 70)
    return human_corpus_path, synthetic_corpus_path

human_corpus_file, synthetic_corpus_file = resolve_data_paths()


Corpus files not located in standard search paths. Preparing Colab upload target...
Select preprocessed_corpus.jsonl (or .gz) and synthetic_corpus.jsonl (or .gz) to upload:


KeyboardInterrupt: 

In [ ]:
#CELL 2.2 - LOAD PREPROCESSED DATASETS AND VALIDATE EXPERIMENTAL PARTITIONS
def open_text_or_gz(filepath: Path):
    #transparently stream plain text or gzip compressed files
    if str(filepath).endswith(".gz"):
        return gzip.open(filepath, "rt", encoding="utf-8")
    return open(filepath, "r", encoding="utf-8")

def load_corpus_partitions(
    human_path: Path, synthetic_path: Path
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    #stream jsonl datasets and organize records into experimental partitions
    human_records = []
    with open_text_or_gz(human_path) as stream_human:
        for line in stream_human:
            if line.strip():
                human_records.append(json.loads(line))

    synthetic_records = []
    with open_text_or_gz(synthetic_path) as stream_synthetic:
        for line in stream_synthetic:
            if line.strip():
                synthetic_records.append(json.loads(line))

    df_human_all = pd.DataFrame(human_records)
    df_synthetic = pd.DataFrame(synthetic_records)

    #extract subsets according to experimental provenance
    df_human_gt = df_human_all[df_human_all["provenance_type"] == "human_ground_truth"].copy()
    df_human_blind = df_human_all[df_human_all["provenance_type"] == "human_temporal_blind"].copy()
    df_longitudinal = df_human_all[df_human_all["provenance_type"] == "longitudinal_unlabeled"].copy()

    #construct balanced modelling dataset matching synthetic and human ground truth counts
    balance_size = min(len(df_human_gt), len(df_synthetic))
    df_human_balanced = df_human_gt.head(balance_size).copy()
    df_synthetic_balanced = df_synthetic.head(balance_size).copy()

    df_modelling = pd.concat([df_human_balanced, df_synthetic_balanced], ignore_index=True)
    df_modelling["label"] = df_modelling["label"].astype(int)
    df_modelling.reset_index(drop=True, inplace=True)

    print("=" * 70)
    print("DATASET PARTITION AND INTEGRITY SUMMARY")
    print(f"Total Preprocessed Human Records:    {len(df_human_all):,}")
    print(f"  Human Ground Truth (Control S_hum):{len(df_human_gt):,}")
    print(f"  Human Temporal Blind (Audit):      {len(df_human_blind):,}")
    print(f"  Longitudinal Unlabeled (2020-2025):{len(df_longitudinal):,}")
    print(f"Total Synthetic AI Records (S_syn):  {len(df_synthetic):,}")
    print("-" * 70)
    print(f"Balanced Modelling Dataset Size:     {len(df_modelling):,} ({len(df_human_balanced):,} Human vs {len(df_synthetic_balanced):,} AI)")
    print(f"Modelling Labels:                    {dict(df_modelling['label'].value_counts())}")
    print("=" * 70)

    return df_human_all, df_synthetic, df_modelling

df_human_full, df_synthetic, df_modelling = load_corpus_partitions(
    human_corpus_file, synthetic_corpus_file
)


In [ ]:
#CELL 2.3 - INSPECT SAMPLE RECORDS AND SECTION STRUCTURE
def display_record_sample(df: pd.DataFrame, title_label: str) -> None:
    #render structured metadata and text excerpts for representative records
    if df is None or len(df) == 0:
        print(f'WARNING: No records available for {title_label}.')
        return

    sample = df.iloc[0]
    raw_label = sample.get('label')
    label_str = 'Human (0)' if raw_label == 0 else ('AI-Generated (1)' if raw_label == 1 else str(raw_label))

    print('=' * 70)
    print(f'SAMPLE INSPECTION: {title_label}')
    print(f'Paper ID:          {sample.get("paper_id")}')
    print(f'Venue (Conference):{sample.get("conference", sample.get("venue"))}')
    print(f'Publication Year:  {sample.get("publication_year", sample.get("year"))}')
    print(f'Label:             {raw_label} ({label_str})')
    print(f'Section Lengths:   {sample.get("section_lengths")}')
    print('-' * 70)
    print('ABSTRACT EXCERPT:')
    print(str(sample.get('abstract') or '')[:350] + '...')
    print('-' * 70)
    print('INTRODUCTION EXCERPT:')
    print(str(sample.get('introduction') or '')[:350] + '...')
    print('-' * 70)
    print('CONCLUSION EXCERPT:')
    print(str(sample.get('conclusion') or '')[:250] + '...')
    print('=' * 70)

display_record_sample(df_modelling[df_modelling['label'] == 0], 'HUMAN GROUND TRUTH (LABEL 0)')
display_record_sample(df_modelling[df_modelling['label'] == 1], 'SYNTHETIC AI GENERATED (LABEL 1)')
